In [1]:
#Verify A100 GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Fri Jul 17 18:01:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
#Verify PyTorch on A100
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Total GPU memory: 85.09 GB


In [3]:
#Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/msc_deepfake_project'
print(f"Project folder: {DRIVE_PROJECT}")

Mounted at /content/drive
Project folder: /content/drive/MyDrive/msc_deepfake_project


In [4]:
#Install Wan 2.2 dependencies
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece
!pip install -q ftfy einops safetensors

import diffusers, transformers, torch
print(f"diffusers: {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print("\nWan 2.2 dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 150.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
diffusers: 0.39.0
transformers: 5.14.1
torch: 2.11.0+cu128

Wan 2.2 dependencies installed.


In [5]:
#Load Wan 2.2 T2V-A14B on A100
from diffusers import WanPipeline, AutoModel
from transformers import UMT5EncoderModel
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("Loading Wan 2.2 T2V-A14B model on A100...")
print("First-time download is ~30-40 GB and takes 15-25 minutes.\n")

MODEL_ID = "Wan-AI/Wan2.2-T2V-A14B-Diffusers"

# Load the pipeline
pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
)

# Move to A100 — plenty of memory
pipe.to("cuda")

# Keep VAE optimisations for stability
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("\nWan 2.2 T2V-A14B model loaded on A100.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Wan 2.2 T2V-A14B model on A100...
First-time download is ~30-40 GB and takes 15-25 minutes.



model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]


Wan 2.2 T2V-A14B model loaded on A100.
GPU memory used: 69.06 GB
GPU memory available: 85.09 GB


In [6]:
#Generating first Wan 2.2 video (same prompt as LTX and HunyuanVideo)
import torch
from diffusers.utils import export_to_video
from datetime import datetime

# Same prompt as LTX and HunyuanVideo tests for direct comparison
prompt = "A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality"
negative_prompt = "blurry, low quality, distorted, unnatural, deformed"

generator = torch.Generator(device="cuda").manual_seed(42)

print(f"Prompt: {prompt}\n")
print("Generating video with Wan 2.2 T2V-A14B on A100...")

start_time = datetime.now()

video_frames = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=480,
    width=832,        # Wan 2.2 uses 480p at 832x480 as its native resolution
    num_frames=81,    # ~5 seconds at 16fps native
    num_inference_steps=40,
    guidance_scale=5.0,   # Wan default guidance
    generator=generator,
).frames[0]

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\nGeneration complete in {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"Number of frames: {len(video_frames)}")

Prompt: A woman with long brown hair smiling gently at the camera in a sunlit park, natural lighting, soft breeze moving her hair, cinematic quality

Generating video with Wan 2.2 T2V-A14B on A100...


  0%|          | 0/40 [00:00<?, ?it/s]


Generation complete in 752.1 seconds (12.5 minutes)
Number of frames: 81


In [7]:
#Saving Wan output to Drive
from datetime import datetime
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_id = f"wan_001_{timestamp}"

video_path = f'{DRIVE_PROJECT}/videos/generated/wan/{video_id}.mp4'
metadata_path = f'{DRIVE_PROJECT}/metadata/{video_id}.json'

# Export video (Wan native fps is 16 but 24 for consistency with LTX and Hunyuan)
export_to_video(video_frames, video_path, fps=16)

# Save metadata in consistent format
metadata = {
    "video_id": video_id,
    "generator": "Wan 2.2 T2V-A14B",
    "generator_version": "Wan-AI/Wan2.2-T2V-A14B-Diffusers",
    "prompt": prompt,
    "negative_prompt": negative_prompt,
    "seed": 42,
    "width": 832,
    "height": 480,
    "num_frames": 81,
    "fps": 16,
    "num_inference_steps": 40,
    "guidance_scale": 5.0,
    "generation_time_seconds": duration,
    "generation_datetime": timestamp,
    "gpu_used": "A100",
    "optimizations": ["vae_tiling", "vae_slicing"]
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Video saved: {video_path}")
print(f"Metadata saved: {metadata_path}")

Video saved: /content/drive/MyDrive/msc_deepfake_project/videos/generated/wan/wan_001_20260717_182819.mp4
Metadata saved: /content/drive/MyDrive/msc_deepfake_project/metadata/wan_001_20260717_182819.json


In [8]:
#Preview inline
from IPython.display import Video
Video(video_path, embed=True, width=400)

In [9]:
#Downloading to local machine
from google.colab import files
files.download(video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>